# Cp + Regroup - Separate Container Triangles

Two-step pipeline:

1. **Cp** - run the standard pressure-coefficient transform on the wind-tunnel body / probe H5s and write `cp.default.time_series.h5`.
2. **Regroup** - apply `cfdmod.regroup` with a `ByConnectivityGrouping` to detect each container as a connected component, then emit a new geometry + Cp timeseries where each container is its own labelled surface and triangles are reordered per container.

Default aggregation is `per_triangle` (one HDF5 column per parent triangle, reordered so each container is a contiguous block). Switch to `area_weighted_mean` in the regroup config cell if you want one aggregated value per container instead.

## 1. Imports

In [ ]:
import pathlib

import h5py
import numpy as np
from lnas import LnasFormat

from cfdmod import (
    BodyDefinition,
    ByConnectivityGrouping,
    BySizeRoundedPerComponent,
    CpCaseConfig,
    RegroupConfig,
    load_mesh,
    mesh_from_h5,
    run_cp,
    run_regroup,
)
from cfdmod.pressure.parameters import CpConfig


: 

## 2. Configure inputs

Set `DATA_DIR` to wherever the simulation results live. The cell auto-discovers the body H5 (`bodies.*.h5`) and probe H5 (`points.*.h5`) inside it; override the explicit paths below if your filenames differ.

All outputs land in `OUTPUT_DIR` (defaults to `<DATA_DIR>/output_regroup`).

In [ ]:
# --- File paths ----------------------------------------------------------
DATA_DIR = pathlib.Path.home() / "Documents/sim_results/results_container/SN"
OUTPUT_DIR = DATA_DIR / "output_regroup"

# Auto-discover body / probe H5s by the standard naming convention. Override
# explicitly if your filenames differ.
_bodies = sorted(DATA_DIR.glob("bodies.*.h5"))
_probes = sorted(DATA_DIR.glob("points.*.h5"))
BODY_H5 = _bodies[0] if _bodies else None
PROBE_H5 = _probes[0] if _probes else None

# Optional: a fixed-frame reference mesh (.lnas / .stl / .h5 / .xdmf). Leave
# as None to use the body H5's own embedded geometry.
REFERENCE_MESH = None

# --- Cp time window (None = use the body H5's full extent) ---------------
T_MIN = None
T_MAX = None

# --- Cp physics ----------------------------------------------------------
# Required by CpConfig. SIMUL_U_H is the simulation flow velocity used as
# the dynamic-pressure denominator; SIMUL_CHARACTERISTIC_LENGTH is the
# reference length (only used as the L/U time scale when normalize_time
# is True). FLUID_DENSITY defaults to 1 (LBM lattice units).
SIMUL_U_H = 1.0
SIMUL_CHARACTERISTIC_LENGTH = 1.0
FLUID_DENSITY = 1.0
MACROSCOPIC_TYPE = "pressure"   # 'pressure' or 'rho' (LBM density)
REFERENCE_PRESSURE = "probe"    # 'probe' (first probe point) or 'average'

# --- Regroup options -----------------------------------------------------
# Step 1: connectivity split. Triangle clusters smaller than MIN_TRIS are
# dropped (typically loose stray triangles).
MIN_TRIS = 4

# Step 2: subdivide each connected component by target container dimensions.
# Each component is split into max(1, round(extent / target)) cells per axis,
# anchored at that component's own bounding box. Standard 20ft container
# defaults: x=6.34m (length), y=2.58m (width), z=2.6m (height). A single
# container becomes one cell; a row of 5 touching containers becomes 5.
TARGET_SIZE_X = 6.34
TARGET_SIZE_Y = 2.58
TARGET_SIZE_Z = 2.6

# 'sliced'             : geometric 90-degree cuts at every cell boundary
#                        (Ce-style); output triangles never straddle two
#                        cells. Per-fragment HDF5 column inherits its parent
#                        triangle's value. Best for clean per-container
#                        rendering and downstream analysis.
# 'per_triangle'       : no slicing; one HDF5 column per parent triangle,
#                        reordered so each container is a contiguous block.
#                        Faster (no geometric cuts), but boundary triangles
#                        get assigned wholly to one cell by their centroid.
# 'area_weighted_mean' : one aggregated value per container (broadcast).
AGGREGATION = "sliced"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR : {DATA_DIR}")
print(f"BODY_H5  : {BODY_H5}")
print(f"PROBE_H5 : {PROBE_H5}")
print(f"OUTPUT   : {OUTPUT_DIR}")
assert BODY_H5 is not None and BODY_H5.exists(), (
    f"No bodies.*.h5 found in {DATA_DIR}; set BODY_H5 explicitly above."
)
assert PROBE_H5 is not None and PROBE_H5.exists(), (
    f"No points.*.h5 found in {DATA_DIR}; set PROBE_H5 explicitly above."
)


## 3. Inspect the mesh

Quick sanity check: triangle count and bounding box of the geometry the rest of the notebook will reason about.

In [ ]:
if REFERENCE_MESH is not None:
    mesh = load_mesh(REFERENCE_MESH)
    mesh_source = REFERENCE_MESH
else:
    mesh = mesh_from_h5(BODY_H5)
    mesh_source = BODY_H5
verts = mesh.geometry.vertices

print(f"Mesh source : {mesh_source}")
print(f"Triangles   : {len(mesh.geometry.triangles):,}")
print(f"Vertices    : {len(verts):,}")
print(
    f"Bounding box: "
    f"x=[{verts[:,0].min():.2f}, {verts[:,0].max():.2f}], "
    f"y=[{verts[:,1].min():.2f}, {verts[:,1].max():.2f}], "
    f"z=[{verts[:,2].min():.2f}, {verts[:,2].max():.2f}]"
)

## 4. Build the Cp config

Single body covering every surface in the mesh. We leave `statistics` unset (defaults to `[]`), which skips the stats step entirely - this notebook only needs the per-triangle Cp timeseries that regroup consumes downstream. To also write a `stats.h5`, pass e.g. `statistics=[BasicStatisticModel(stats="mean"), BasicStatisticModel(stats="rms")]` (note the field name is `stats`, plural).


In [ ]:
# Resolve the time window against the body H5's actual extent.
def _detect_time_range(body_h5: pathlib.Path) -> tuple[float, float]:
    with h5py.File(body_h5, "r") as f:
        keys = list(f["pressure"].keys())
    times = sorted(float(k[1:]) for k in keys)
    return float(times[0]), float(times[-1])


_detected = _detect_time_range(BODY_H5)
timestep_range = (
    _detected[0] if T_MIN is None else max(float(T_MIN), _detected[0]),
    _detected[1] if T_MAX is None else min(float(T_MAX), _detected[1]),
)
print(f"Body H5 time range : [{_detected[0]:.3f}, {_detected[1]:.3f}]")
print(f"Cropped to         : [{timestep_range[0]:.3f}, {timestep_range[1]:.3f}]")

cp_cfg = CpCaseConfig(
    pressure_coefficient={
        "default": CpConfig(
            timestep_range=timestep_range,
            simul_U_H=SIMUL_U_H,
            simul_characteristic_length=SIMUL_CHARACTERISTIC_LENGTH,
            fluid_density=FLUID_DENSITY,
            macroscopic_type=MACROSCOPIC_TYPE,
            reference_pressure=REFERENCE_PRESSURE,
        )
    }
)


## 5. Run Cp

Streams the body / probe H5s and writes `cp.default.time_series.{h5,xdmf}`. With `statistics=[]` (the default) no `stats.h5` is produced; add stats to the Cp config above if you also want one.


In [ ]:
run_cp(
    body_h5=BODY_H5,
    probe_h5=PROBE_H5,
    cfg_path=cp_cfg,
    output=OUTPUT_DIR,
    mesh_path=REFERENCE_MESH,
)
CP_TIMESERIES = OUTPUT_DIR / "cp.default.time_series.h5"
print(f"Cp timeseries written to {CP_TIMESERIES}")

## 6. Regroup - separate containers, then subdivide by container dimensions

Two-step chain:

1. **`ByConnectivityGrouping`** - label each connected component as a parent group.
2. **`BySizeRoundedPerComponent`** - within each component, subdivide by the standard container dimensions (`TARGET_SIZE_X/Y/Z` from the inputs cell). Each axis gets `max(1, round(extent / target))` cells anchored at that component's own bbox, so a row of touching containers gets split into one cell per container.

The connectivity components are scaffolding and are dropped from the final output; only the leaf cells (one per container) are written. Output:

- `regroup_out/geometry.lnas` - one named surface per container cell.
- `regroup_out/cp.regrouped.{h5,xdmf}` - the Cp timeseries with columns reordered (per_triangle) or aggregated (area_weighted_mean) per cell.


In [ ]:
regroup_cfg = RegroupConfig(
    groupings=[
        ByConnectivityGrouping(
            name_template="cc{idx}",
            min_triangles=MIN_TRIS,
        ),
        BySizeRoundedPerComponent(
            target_size_x=TARGET_SIZE_X,
            target_size_y=TARGET_SIZE_Y,
            target_size_z=TARGET_SIZE_Z,
            name_template="{parent}_c{idx}",
        ),
    ],
    aggregation=AGGREGATION,
    timeseries_group="cp",
    output_geometry_format="lnas",
    unassigned_policy="drop",
)

REGROUP_DIR = OUTPUT_DIR / "regroup_out"
run_regroup(
    cfg=regroup_cfg,
    geometry=mesh,             # reuse the mesh we loaded above
    timeseries=CP_TIMESERIES,
    output_dir=REGROUP_DIR,
)
print(f"\nRegrouped outputs in {REGROUP_DIR}:")
for p in sorted(REGROUP_DIR.iterdir()):
    print(f"  {p.name}")


## 7. Inspect the regrouped output

List the containers (one named surface each), with triangle counts and per-container bounding boxes. The `region_labels` array in `/meta` gives the container assignment of every output triangle in column order.

In [ ]:
regrouped_lnas = LnasFormat.from_file(REGROUP_DIR / "geometry.lnas")
print(f"Containers detected: {len(regrouped_lnas.surfaces)}")
print(f"Total triangles   : {regrouped_lnas.geometry.triangles.shape[0]:,}\n")

tri_verts = regrouped_lnas.geometry.triangle_vertices
for name, idxs in sorted(regrouped_lnas.surfaces.items()):
    sub = tri_verts[idxs]
    lo = sub.reshape(-1, 3).min(axis=0)
    hi = sub.reshape(-1, 3).max(axis=0)
    print(
        f"  {name:<20s} {len(idxs):>6d} tris  "
        f"x=[{lo[0]:7.2f}, {hi[0]:7.2f}]  "
        f"y=[{lo[1]:7.2f}, {hi[1]:7.2f}]  "
        f"z=[{lo[2]:7.2f}, {hi[2]:7.2f}]"
    )

with h5py.File(REGROUP_DIR / "cp.regrouped.h5", "r") as f:
    n_steps = sum(1 for k in f["cp"].keys() if k.startswith("t"))
    n_cols = f["cp"][next(k for k in f["cp"].keys() if k.startswith("t"))].shape[0]
    region_labels = [s.decode() for s in f["meta/region_labels"][:]]
print(f"\nCp timeseries: {n_steps} timestep(s), {n_cols} column(s) per step")
print(f"region_labels length: {len(region_labels)} (one per output triangle)")

## 9. Open in ParaView

| File | What it shows |
|---|---|
| `output_regroup/cp.default.time_series.xdmf` | The original Cp animation on the full unsegmented mesh. |
| `output_regroup/regroup_out/cp.regrouped.xdmf` | The same Cp animation, but the geometry is now sliced per container (one labelled surface per cell). Use ParaView's **Extract Block** filter to isolate a single container by name. |
| `output_regroup/regroup_out/cp.per_face.xdmf` | The Cp animation reduced to one value per `(container, face_normal)` - each container face shows up as a single flat color per timestep. Pair with `geometry.per_face.lnas` if you want **Extract Block** by face (`<cell>_zp`, `<cell>_xn`, ...). |


In [ ]:
from cfdmod.io.xdmf import write_temporal_xdmf

SLICED_H5 = REGROUP_DIR / "cp.regrouped.h5"
SLICED_LNAS = REGROUP_DIR / "geometry.lnas"
PER_FACE_H5 = REGROUP_DIR / "cp.per_face.h5"
PER_FACE_XDMF = REGROUP_DIR / "cp.per_face.xdmf"
PER_FACE_LNAS = REGROUP_DIR / "geometry.per_face.lnas"

sliced = LnasFormat.from_file(SLICED_LNAS)
n_tri = sliced.geometry.triangles.shape[0]

# Per-fragment areas + axis-aligned direction (0=+x, 1=-x, 2=+y, 3=-y, 4=+z, 5=-z).
tri_v = sliced.geometry.triangle_vertices
crosses = np.cross(tri_v[:, 1] - tri_v[:, 0], tri_v[:, 2] - tri_v[:, 0])
areas = np.linalg.norm(crosses, axis=1) / 2
norms = crosses / (np.linalg.norm(crosses, axis=1, keepdims=True) + 1e-30)
axis_idx = np.abs(norms).argmax(axis=1)
sign_neg = norms[np.arange(n_tri), axis_idx] < 0
direction = (axis_idx * 2 + sign_neg.astype(np.int64)).astype(np.int64)

# Per-fragment cell index, derived from the sliced surfaces dict.
cell_names = sorted(sliced.surfaces.keys())
cell_of = np.empty(n_tri, dtype=np.int64)
for ci, name in enumerate(cell_names):
    cell_of[sliced.surfaces[name]] = ci

# Group key: (cell_idx, direction) -> a unique int.
group_key = cell_of * 6 + direction
unique_groups, inverse = np.unique(group_key, return_inverse=True)

# Per-group total area (used for area-weighted mean weights).
group_total_area = np.zeros(len(unique_groups), dtype=np.float64)
np.add.at(group_total_area, inverse, areas)
weights = areas / np.where(group_total_area[inverse] > 0, group_total_area[inverse], 1.0)

print(
    f"{n_tri:,} fragments -> {len(unique_groups):,} (container, face) buckets "
    f"({len(cell_names)} containers x up to 6 faces)"
)

# Stream-aggregate per timestep.
with h5py.File(SLICED_H5, "r") as src, h5py.File(PER_FACE_H5, "w") as dst:
    dst.create_dataset("Triangles", data=src["Triangles"][:])
    dst.create_dataset("Geometry", data=src["Geometry"][:])
    meta_src = src["meta"]
    meta_dst = dst.create_group("meta")
    for k in ("time_steps", "time_normalized"):
        meta_dst.create_dataset(k, data=meta_src[k][:])

    cp_dst = dst.create_group("cp")
    keys = sorted(k for k in src["cp"].keys() if k.startswith("t"))
    for key in keys:
        in_col = src["cp"][key][:]
        weighted = in_col * weights
        per_group = np.zeros(len(unique_groups), dtype=np.float64)
        np.add.at(per_group, inverse, weighted)
        cp_dst.create_dataset(key, data=per_group[inverse].astype(np.float64))

write_temporal_xdmf(PER_FACE_H5, PER_FACE_XDMF, "cp")

# Companion geometry.per_face.lnas: same triangles, surfaces renamed by face.
direction_suffix = {0: "xp", 1: "xn", 2: "yp", 3: "yn", 4: "zp", 5: "zn"}
new_surfaces = {}
for gi, gkey in enumerate(unique_groups):
    cell_idx = int(gkey // 6)
    dir_idx = int(gkey % 6)
    name = f"{cell_names[cell_idx]}_{direction_suffix[dir_idx]}"
    new_surfaces[name] = np.flatnonzero(inverse == gi).astype(np.int32)

per_face_lnas = LnasFormat(
    version=sliced.version,
    geometry=sliced.geometry,
    surfaces=new_surfaces,
)
per_face_lnas.to_file(PER_FACE_LNAS)

print(f"\nWrote: {PER_FACE_H5.name}, {PER_FACE_XDMF.name}, {PER_FACE_LNAS.name}")
print(f"Per-face surfaces: {len(new_surfaces):,}")


## 8. Aggregate per face: one Cp value per (container, normal direction)

The sliced output has one Cp column per *fragment*. To get a single Cp value per **container face** (top, bottom, four sides), we group fragments by (container_cell, axis-aligned normal direction) and area-weight-average their Cp values inside each bucket.

Each fragment's normal is quantised to one of 6 directions:

- `0` = +x, `1` = -x
- `2` = +y, `3` = -y
- `4` = +z, `5` = -z

Output:

- `regroup_out/cp.per_face.{h5,xdmf}` - same geometry as the sliced output, but the per-fragment Cp at each timestep is replaced by its (container, face) area-weighted mean (broadcast back to every fragment in the group). Visually each container face becomes a single flat color.
- `regroup_out/geometry.per_face.lnas` - same triangles, but surfaces are renamed `<cell>_d{0..5}` so ParaView's **Extract Block** can isolate any single face of any container.
